# JP-1：引き直しの長さを数える（日本語Aの結果・CPU だけ）

日本語A の本番は 16.5 時間かかり、見込み（1.6 時間）を大きく超えました。計画の止める線（全体 45 CU）に当たったので、続け方を決める前に、**時間がどこに使われたか**を数えます。

- Drive の `DECISION_POINTS/jp1/MAIN/JP1_ja_A/` の結果を**読むだけ**です（書き換えません）。ファイルはすべて中身のハッシュで確かめてから使います。
- 数えるのは、引き直しが EOS で止まったか 512 の上限まで走ったか、32本の組がいつ終わったか、長さ（数だけ）、作業のログの1単位あたりの秒数です。
- GPU もモデルも使いません。MGSM は読まず、トークンの中身も出しません。

**使い方**：ランタイムは **CPU（GPU なし）** にしてください。シークレットは `GITHUB_TOKEN`。「すべてのセルを実行」で **数分**。結果は Drive の `DECISION_POINTS/jp1/LENGTHCOUNT/`。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

In [ ]:
# Cell 1 — Drive, CPU runtime, exact head, notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'JP-1 引き直しの長さを数える（日本語Aの結果・CPU・数だけ）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - JP-1 length count', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'jp1_lengthcount'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('⚠ 止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

drive.mount("/content/drive")
EXPECTED_HEAD = "cdf77e1861d1356ad3ec37ce073a3691d4ce92ab"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None
token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
THEME = Path("/content/drive/MyDrive/DECISION_POINTS")
SRC = THEME / "jp1" / "MAIN" / "JP1_ja_A"       # read only
OUT = THEME / "jp1" / "LENGTHCOUNT"
OUT.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))
STARTED = time.time()
print("head:", head)

In [ ]:
# Cell 2 — read and validate every unit file (envelope hash and identity), count, write numbers only.
from pork import jp1 as P
from pork import jp1_runner as rr
from pork import jp1_length_count as LC
from pork.quality_pork_q3a_runner import atomic_json
ID = json.loads((SRC / "RUN_IDENTITY.json").read_text(encoding="utf-8"))
LANG, MODEL, COMMIT, ITEMS_SHA = ID["lang"], ID["model"], ID["code_commit"], ID["items_sha256"]
bases, points = {}, {}
for p in sorted((SRC / "units").glob("*.json")):
    parts = p.stem.split("_")
    kind, row = parts[0], int(parts[1])
    i = int(parts[2]) if kind == "point" else None
    body = rr.read_envelope(p, LANG, MODEL, kind, row, i, COMMIT, ITEMS_SHA, tries=1)
    if kind == "base":
        bases[row] = body
    else:
        points[(row, i)] = body
SECONDS = LC.parse_logs(p.read_text(errors="replace") for p in sorted(SRC.glob("worker_*.log")))
RES = LC.summarize(bases, points, SECONDS, workers=P.MAX_GPU_WORKERS)
RES.update({"source": "jp1/MAIN/JP1_ja_A", "run_code_commit": COMMIT, "head": EXPECTED_HEAD, "lang": LANG,
            "finished_jst": dt.datetime.now(JST).isoformat(timespec="seconds")})
del bases, points
atomic_json(OUT / "LENGTH_COUNT_ja_A.json", RES)
sha_ = hashlib.sha256((OUT / "LENGTH_COUNT_ja_A.json").read_bytes()).hexdigest()
print("LENGTH_COUNT_ja_A.json sha256:", sha_, "| bytes:", (OUT / "LENGTH_COUNT_ja_A.json").stat().st_size)
print(json.dumps({"points": RES["points"], "steps_total": RES["steps_total"], "assumed_steps_total": RES["assumed_steps_total"],
                  "fit": RES["fit"], "any_row_to_budget": {k: v["batches_any_row_to_budget_share"] for k, v in RES["kinds"].items()},
                  "caps_hours": {c: round(v["wall_hours_if_c_holds"], 2) for c, v in RES["caps"].items()}}, indent=1))
notify("🎉 引き直しの長さを数え終わりました（結果は Drive の jp1/LENGTHCOUNT）", tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()